# Yapay Sinir Ağları — AŞAMA 2: Nested CV (Seed **31040**)

**Yeni ve bağımsız çalışma, 18 Kasım 2026.** Aşama 1'in Google Drive'daki `YSA_31040_20261118` klasörünü kullanır. Aşama 1'in `dataset_split.csv`, `seeds.json` ve ham veri SHA-256 değerleri **değiştirilmez**.

**Kural:** Test satırları model eğitimi, hiperparametre seçimi veya metrik hesaplaması için **asla kullanılmaz**. Tüm değerlendirmeler yalnızca development verisinde yapılır.

- 3 outer × 3 inner × (6 manuel + 6 otomatik) = **108 iç doğrulama eğitimi**.
- 3 outer × 2 yöntem = **6 dış-kat yeniden eğitim**; ayrıca her dış katta Dummy + Logistic Regression.
- Tam development üzerinde 3 inner × 12 = **36 nihai seçim eğitimi**.
- Sızıntısız preprocessing: sadece ağırlık güncellemesi yapılan **fit** alt kümesinde öğrenilir; **%15 early-stop monitor**, inner validation ve outer evaluation yalnızca dönüştürülür.
- Erken durdurma BCE kaybına; hiperparametre seçimi inner validation **Average Precision (AP)** skoruna dayanır.

**Çalıştırma:** Colab → Çalışma Zamanı → Tümünü Çalıştır. Çok sayıda eğitim nedeniyle uzun sürebilir; her biten deney Drive'a yazılır. Bağlantı koparsa **aynı notebook'u ve aynı Drive klasörünü** yeniden çalıştırın; tamamlanan kayıtlar denetlenerek yeniden kullanılır.

## 0. Kütüphaneler, çalışma klasörü ve kayıt yardımcıları

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, json, time, copy, random, hashlib, warnings, platform, traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix)
from IPython.display import display

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/Yapay Sinir Ağları/YSA_31040_20261118')
else:
    ROOT=Path.cwd()/'YSA_31040_20261118'
DATA, RESULTS, CONFIGS, FIGURES = [ROOT/p for p in ('data','results','configs','figures')]
RUNS=RESULTS/'stage2_runs'
HISTORIES=RESULTS/'stage2_histories'
for folder in (RUNS,HISTORIES,CONFIGS,FIGURES):folder.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CODE_VERSION='stage2-seed31040-v1'

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def stable_hash(obj):
    encoded=json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False)
    return hashlib.sha256(encoded.encode('utf-8')).hexdigest()
def save_json_atomic(path,obj):
    path=Path(path)
    tmp=path.with_name(path.name+'.part')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    tmp.replace(path)
def freeze_json(path,obj):
    path=Path(path)
    if path.exists():
        previous=json.loads(path.read_text(encoding='utf-8'))
        if previous!=obj:raise RuntimeError('KİLİTLİ KONFİGÜRASYON DEĞİŞMİŞ: '+str(path))
    else:
        save_json_atomic(path,obj)
def save_csv_atomic(frame,path):
    path=Path(path);tmp=path.with_name(path.name+'.part')
    frame.to_csv(tmp,index=False);tmp.replace(path)
print('Çalışma alanı:',ROOT)
print('Cihaz:',DEVICE,'| Python:',sys.version.split()[0], '| PyTorch:',torch.__version__,'| sklearn:',sklearn.__version__)

## 1. Aşama 1 verisini doğrulama ve **yalnızca development** bölümünü alma

Bu kod Aşama 1'in dosya bütünlüğünü SHA-256 ile denetler. EDA çıktılarının test kilidi olmadığına dikkat edin: Aşama 3 test tahminlerini daha sonra **ayrı bir commit** ile sabitleyeceğiz.

In [ ]:
needed=[DATA/'bank-additional.csv',RESULTS/'dataset_split.csv',
        RESULTS/'raw_data_sha256.json',RESULTS/'stage1_sha256_manifest.json',
        CONFIGS/'seeds.json',CONFIGS/'eda_hypotheses.json']
missing=[str(p) for p in needed if not p.is_file()]
if missing:raise FileNotFoundError('Önce Aşama 1 tamamlanmalı. Eksik dosyalar: '+str(missing))
SEEDS=json.loads((CONFIGS/'seeds.json').read_text(encoding='utf-8'))
assert SEEDS['base']==31040 and SEEDS['split']==31093, 'Seed uyuşmazlığı'
raw_meta=json.loads((RESULTS/'raw_data_sha256.json').read_text(encoding='utf-8'))
assert raw_meta['sha256']==sha256_file(DATA/'bank-additional.csv'),'Ham veri SHA-256 değişmiş'
manifest=json.loads((RESULTS/'stage1_sha256_manifest.json').read_text(encoding='utf-8'))
for rel in ('results/dataset_split.csv','configs/seeds.json','configs/eda_hypotheses.json'):
    assert manifest['files'][rel]==sha256_file(ROOT/rel),'Aşama 1 dosyası değiştirilmiş: '+rel
raw=pd.read_csv(DATA/'bank-additional.csv',sep=';')
assert raw.shape==(4119,21) and set(raw['y'].unique())=={'yes','no'}
# Aşama 1'deki tam kopya politikasını ve indeksleri aynı sırada uygula.
raw=raw.loc[~raw.duplicated(keep='first')].copy()
split=pd.read_csv(RESULTS/'dataset_split.csv')
assert split['original_index'].is_unique and set(split['split'].unique())=={'development','test'}
assert set(split['original_index'].astype(int))==set(raw.index), 'Split/ham veri satır indeksi uyuşmuyor'
DEV_IDS=split.loc[split['split']=='development','original_index'].astype(int).to_numpy()
DEV=raw.loc[DEV_IDS].copy()
X=DEV.drop(columns=['y','duration'])
y=DEV['y'].map({'no':0,'yes':1}).astype(int).to_numpy()
assert len(np.unique(y))==2 and 'duration' not in X.columns
NUM=X.select_dtypes(include='number').columns.tolist()
CAT=X.select_dtypes(exclude='number').columns.tolist()
SPLIT_SHA=sha256_file(RESULTS/'dataset_split.csv')
RAW_SHA=sha256_file(DATA/'bank-additional.csv')
print('Aşama 1 bütünlük kontrolü BAŞARILI')
print('Development satırları:',len(y),'| pozitif:',int(y.sum()),'| özellik:',X.shape[1])
print('Bağımsız test satırlarına burada fit/transform veya skor uygulanmayacak.')

## 2. Sızıntısız ön işleme, MLP eğitimi, scheduler ve early stopping

**Sabit eğitim koşulları:** tek logit çıkışı, `BCEWithLogitsLoss`, `sigmoid`, en fazla 150 epoch, early stopping patience=15, scheduler factor=0.5/patience=5/min_lr=1e-5. Optimizer: SGD+Momentum(0.9), Adam veya AdamW.

Sayısal eksikler medyan ile doldurulur; kategorik eksikler en sık kategoriyle, `unknown` ise **ayrı kategori olarak** kalır. `pdays=999` özel kodu clipping ile değiştirilmez. Tüm dönüşümler ilgili fold'un **ağırlık eğitimi alt kümesinde** `fit` edilir.

In [ ]:
class PercentileClipper(BaseEstimator,TransformerMixin):
    def __init__(self,column_names=(),enabled=False,q_low=0.01,q_high=0.99):
        self.column_names=column_names
        self.enabled=enabled
        self.q_low=q_low
        self.q_high=q_high
    def fit(self,Z,y=None):
        a=np.asarray(Z,dtype=float)
        self.lo_=np.full(a.shape[1],-np.inf)
        self.hi_=np.full(a.shape[1],np.inf)
        if self.enabled:
            for j,name in enumerate(self.column_names):
                if name=='pdays':continue  # 999 önceki temas yok; olağan aykırı değer değil.
                self.lo_[j]=np.quantile(a[:,j],self.q_low)
                self.hi_[j]=np.quantile(a[:,j],self.q_high)
        return self
    def transform(self,Z):
        return np.clip(np.asarray(Z,dtype=float),self.lo_,self.hi_)

def make_preprocessor(cfg):
    scaler={'standard':StandardScaler,'minmax':MinMaxScaler,'robust':RobustScaler}[cfg['scaler']]()
    num=Pipeline([('imputer',SimpleImputer(strategy='median')),
                  ('clip',PercentileClipper(tuple(NUM),cfg['clipping'])),
                  ('scale',scaler)])
    try:onehot=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError:onehot=OneHotEncoder(handle_unknown='ignore',sparse=False)
    cat=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                  ('onehot',onehot)])
    return ColumnTransformer([('num',num,NUM),('cat',cat,CAT)],sparse_threshold=0)

def make_model(d,cfg):
    widths=[d]+list(cfg['hidden'])
    layers=[]
    for a,b in zip(widths[:-1],widths[1:]):
        layers.extend([nn.Linear(a,b),nn.ReLU() if cfg['activation']=='relu' else nn.Tanh()])
        if cfg['dropout']>0:layers.append(nn.Dropout(cfg['dropout']))
    layers.append(nn.Linear(widths[-1],1))
    return nn.Sequential(*layers)

def set_seed(seed):
    random.seed(int(seed));np.random.seed(int(seed));torch.manual_seed(int(seed))
    if torch.cuda.is_available():torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True,warn_only=True)

def metrics(truth,prob):
    truth=np.asarray(truth,dtype=int);prob=np.asarray(prob,dtype=float)
    prediction=(prob>=0.50).astype(int)
    tn,fp,fn,tp=confusion_matrix(truth,prediction,labels=[0,1]).ravel()
    return dict(ap=float(average_precision_score(truth,prob)),
        roc_auc=float(roc_auc_score(truth,prob)),
        accuracy=float(accuracy_score(truth,prediction)),
        precision=float(precision_score(truth,prediction,zero_division=0)),
        sensitivity=float(recall_score(truth,prediction,zero_division=0)),
        specificity=float(tn/(tn+fp)) if tn+fp else 0.,
        f1=float(f1_score(truth,prediction,zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(truth,prediction)),
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

def fit_predict(train_ids,eval_ids,cfg,seed,history_path):
    t0=time.perf_counter()
    set_seed(seed)
    y_train_all=y[train_ids]
    fit_local,monitor_local=train_test_split(np.arange(len(train_ids)),
        test_size=0.15,stratify=y_train_all,random_state=int(seed))
    fit_ids=np.asarray(train_ids)[fit_local]
    monitor_ids=np.asarray(train_ids)[monitor_local]
    prep=make_preprocessor(cfg)
    Z_fit=np.asarray(prep.fit_transform(X.iloc[fit_ids]),dtype=np.float32)
    Z_monitor=np.asarray(prep.transform(X.iloc[monitor_ids]),dtype=np.float32)
    Z_eval=np.asarray(prep.transform(X.iloc[eval_ids]),dtype=np.float32)
    model=make_model(Z_fit.shape[1],cfg).to(DEVICE)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if cfg['optimizer']=='sgd':
        optim=torch.optim.SGD(model.parameters(),lr=cfg['lr'],momentum=0.9,weight_decay=cfg['weight_decay'])
    elif cfg['optimizer']=='adam':
        optim=torch.optim.Adam(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
    else:
        optim=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
    scheduler=(torch.optim.lr_scheduler.ReduceLROnPlateau(
        optim,mode='min',factor=0.5,patience=5,min_lr=1e-5)
        if cfg['scheduler'] else None)
    criterion=nn.BCEWithLogitsLoss()
    loader=DataLoader(TensorDataset(torch.from_numpy(Z_fit),
        torch.tensor(y[fit_ids],dtype=torch.float32)),
        batch_size=cfg['batch_size'],shuffle=True,
        generator=torch.Generator().manual_seed(int(seed)),num_workers=0)
    mon_X=torch.from_numpy(Z_monitor).to(DEVICE)
    mon_y=torch.tensor(y[monitor_ids],dtype=torch.float32,device=DEVICE)
    best_loss=float('inf');best_epoch=0;best_state=None;epochs_without_improvement=0
    history=[]
    for epoch in range(1,151):
        model.train();loss_sum=0.;n_seen=0
        for xb,yb in loader:
            xb=xb.to(DEVICE);yb=yb.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            loss=criterion(model(xb).flatten(),yb)
            if not torch.isfinite(loss):raise RuntimeError('Non-finite training BCE')
            loss.backward();optim.step()
            loss_sum+=float(loss.item())*len(yb);n_seen+=len(yb)
        model.eval()
        with torch.no_grad():monitor_loss=float(criterion(model(mon_X).flatten(),mon_y).item())
        if not np.isfinite(monitor_loss):raise RuntimeError('Non-finite early stop BCE')
        current_lr=float(optim.param_groups[0]['lr'])
        history.append(dict(epoch=epoch,training_bce=loss_sum/n_seen,
                            monitor_bce=monitor_loss,learning_rate=current_lr))
        if monitor_loss<best_loss-1e-8:
            best_loss=monitor_loss;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
            epochs_without_improvement=0
        else:epochs_without_improvement+=1
        if scheduler is not None:scheduler.step(monitor_loss)
        if epochs_without_improvement>=15:break
    model.load_state_dict(best_state)
    model.eval();parts=[]
    with torch.no_grad():
        for i in range(0,len(Z_eval),1024):
            tensor=torch.from_numpy(Z_eval[i:i+1024]).to(DEVICE)
            parts.append(torch.sigmoid(model(tensor).flatten()).cpu().numpy())
    save_csv_atomic(pd.DataFrame(history),history_path)
    return np.concatenate(parts),dict(best_epoch=int(best_epoch),epochs_run=len(history),
        monitor_best_bce=float(best_loss),train_seconds=float(time.perf_counter()-t0),
        parameter_count=int(n_params),monitor_metric='BCE',
        history_path=str(history_path.relative_to(ROOT)))
print('MLP ve eğitim fonksiyonları hazır.')

## 3. 6 manuel + 6 otomatik adayın **önceden** oluşturulup kilitlenmesi

M1–M6 doğrudan yönergenin tablolarından alınmıştır. SGD'nin M3'te `lr=0.001` kullanması yavaş yakınsamaya neden olabilir; bu durum raporda tartışılacak. R1–R6 deterministik rastgele arama ile **her optimizer için tam 2 aday** üretir. Aday listeleri dış kat sonuçlarına göre değiştirilmez.

In [ ]:
BASE_CFG=dict(optimizer='adamw',hidden=[32],scaler='standard',lr=0.001,
    weight_decay=1e-4,batch_size=64,activation='relu',dropout=0.,
    scheduler=True,clipping=False)
changes=[{}, {'optimizer':'adam'}, {'optimizer':'sgd'},
         {'scaler':'minmax'}, {'hidden':[64,32]}, {'scheduler':False}]
manual=[]
for i,change in enumerate(changes,1):
    manual.append({**BASE_CFG,**change,'id':f'M{i}'})
rng=np.random.default_rng(SEEDS['base']+173)
random_candidates=[]
for optimizer in ('sgd','adam','adamw'):
    for _ in range(2):
        cfg=dict(id=f'R{len(random_candidates)+1}',optimizer=optimizer,
            hidden=list(((16,),(32,),(64,),(64,32),(128,64))[int(rng.integers(5))]),
            activation=('relu','tanh')[int(rng.integers(2))],
            lr=float(10**rng.uniform(-3,-1)) if optimizer=='sgd' else float(10**rng.uniform(-4,-2)),
            weight_decay=float((0,1e-5,1e-4,1e-3)[int(rng.integers(4))]),
            batch_size=int((32,64,128)[int(rng.integers(3))]),
            scaler=('standard','minmax','robust')[int(rng.integers(3))],
            dropout=float((0.0,0.2)[int(rng.integers(2))]),
            clipping=bool(rng.integers(2)),scheduler=True)
        random_candidates.append(cfg)
assert len(manual)==len(random_candidates)==6
assert sorted(q['optimizer'] for q in random_candidates)==['adam','adam','adamw','adamw','sgd','sgd']
freeze_json(CONFIGS/'manual_candidates.json',manual)
freeze_json(CONFIGS/'random_candidates.json',random_candidates)
training_policy={'max_epochs':150,'early_stop_patience':15,
    'monitor_fraction':0.15,'scheduler_factor':0.5,'scheduler_patience':5,
    'min_lr':1e-5,'selection_metric':'average_precision','early_stop_metric':'BCE',
    'threshold':0.5,'outer_folds':3,'inner_folds':3,'base_seed':31040}
freeze_json(CONFIGS/'training_policy.json',training_policy)
PLAN_ID=stable_hash({'code_version':CODE_VERSION,'manual':manual,'random':random_candidates,
                     'training_policy':training_policy,'split_sha256':SPLIT_SHA,'raw_sha256':RAW_SHA})
freeze_json(CONFIGS/'stage2_plan.json',{'plan_sha256':PLAN_ID,
    'code_version':CODE_VERSION,'split_sha256':SPLIT_SHA,'raw_sha256':RAW_SHA,
    'manual_n':6,'random_n':6,'base_seed':31040})
ALL_CANDIDATES={q['id']:q for q in manual+random_candidates}
display(pd.DataFrame(manual))
display(pd.DataFrame(random_candidates))
print('Plan kilitlendi, SHA-256:',PLAN_ID)

## 4. Eğitim kayıtları, önbellek, inner CV ve referans modeller

Her deney tamamlanır tamamlanmaz JSON ve eğitim geçmişi CSV olarak kaydedilir. Yeniden çalıştırılırsa **önceden kaydedilen konfigürasyon, indeks ve seed imzası** kontrol edilir. Uyuşmazlık varsa devam etmez. Başarısız bir eğitimde hata kaydı saklanır; sonuç uydurulmaz.

In [ ]:
def seed_for(phase,inner_fold=0,variant=0):
    # Aynı aşama/katta adaylar ortak başlangıç seed politikasını kullanır.
    code=10000 if phase=='final' else 1000*int(phase.replace('outer',''))
    return int(SEEDS['torch']+code+100*inner_fold+variant)

def cached_train(key,cfg,train_idx,eval_idx,seed,method):
    path=RUNS/(key+'.json')
    hist_path=HISTORIES/(key+'.csv')
    signature=stable_hash({'plan':PLAN_ID,'key':key,'config':cfg,
        'seed':int(seed),'train_ids':np.asarray(train_idx).astype(int).tolist(),
        'eval_ids':np.asarray(eval_idx).astype(int).tolist()})
    if path.exists():
        result=json.loads(path.read_text(encoding='utf-8'))
        if result.get('signature')!=signature:
            raise RuntimeError('Cache içerik/seed/split uyuşmazlığı: '+key)
        if not hist_path.exists() or sha256_file(hist_path)!=result['history_sha256']:
            raise RuntimeError('Eğitim geçmişi kayıp/değişmiş: '+key)
        return result
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            probs,info=fit_predict(train_idx,eval_idx,cfg,seed,hist_path)
        warning_text=list(dict.fromkeys(str(w.message) for w in caught))
        result=dict(key=key,method=method,candidate=cfg['id'],signature=signature,
            seed=int(seed),train_n=int(len(train_idx)),eval_n=int(len(eval_idx)),
            history_sha256=sha256_file(hist_path),warnings=warning_text,
            **info,**metrics(y[eval_idx],probs))
        save_json_atomic(path,result)
        return result
    except Exception as e:
        save_json_atomic(RUNS/(key+'.failure.json'),{'key':key,'exception':repr(e),
            'traceback':traceback.format_exc(),'plan_sha256':PLAN_ID})
        raise

def run_inner(phase,outer_train_idx,inner_seed):
    outer_train_idx=np.asarray(outer_train_idx,dtype=int)
    folds=list(StratifiedKFold(n_splits=3,shuffle=True,random_state=int(inner_seed))
        .split(X.iloc[outer_train_idx],y[outer_train_idx]))
    records=[]
    for method,candidates in [('manual',manual),('random',random_candidates)]:
        for cfg in candidates:
            each=[]
            for k,(train_local,val_local) in enumerate(folds,1):
                train_idx=outer_train_idx[train_local];val_idx=outer_train_idx[val_local]
                record=cached_train(f'{phase}_{method}_{cfg["id"]}_inner{k}',cfg,
                    train_idx,val_idx,seed_for(phase,k),method)
                each.append(record)
            records.append(dict(phase=phase,method=method,candidate=cfg['id'],
                mean_ap=float(np.mean([r['ap'] for r in each])),
                sd_ap=float(np.std([r['ap'] for r in each],ddof=1)),
                total_seconds=float(sum(r['train_seconds'] for r in each)),
                mean_best_epoch=float(np.mean([r['best_epoch'] for r in each])),
                fold_1_ap=each[0]['ap'],fold_2_ap=each[1]['ap'],fold_3_ap=each[2]['ap'],
                config_sha256=stable_hash(cfg)))
            print(phase,method,cfg['id'],'mean AP:',round(records[-1]['mean_ap'],4),flush=True)
        # Her metodun ara sonucu ayrı saklanır.
        save_csv_atomic(pd.DataFrame([z for z in records if z['method']==method]),
                        RESULTS/f'{phase}_{method}_search.csv')
    result=pd.DataFrame(records)
    save_csv_atomic(result,RESULTS/f'{phase}_inner_search.csv')
    return result

def baseline_outer(kind,train_idx,eval_idx,phase):
    key=f'{phase}_baseline_{kind}';path=RUNS/(key+'.json')
    signature=stable_hash({'plan':PLAN_ID,'key':key,
        'train':np.asarray(train_idx).astype(int).tolist(),
        'eval':np.asarray(eval_idx).astype(int).tolist()})
    if path.exists():
        obj=json.loads(path.read_text(encoding='utf-8'))
        if obj.get('signature')!=signature:raise RuntimeError('Referans cache uyuşmazlığı')
        return obj
    t0=time.perf_counter()
    pp=make_preprocessor(BASE_CFG)
    tr=pp.fit_transform(X.iloc[train_idx]);ev=pp.transform(X.iloc[eval_idx])
    model=DummyClassifier(strategy='prior') if kind=='dummy' else LogisticRegression(
        max_iter=2000,random_state=SEEDS['base'])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        model.fit(tr,y[train_idx])
    prob=model.predict_proba(ev)[:,list(model.classes_).index(1)]
    obj=dict(key=key,method=kind,candidate=kind,signature=signature,
        train_n=int(len(train_idx)),eval_n=int(len(eval_idx)),
        train_seconds=float(time.perf_counter()-t0),
        warnings=list(dict.fromkeys(str(w.message) for w in caught)),
        **metrics(y[eval_idx],prob))
    save_json_atomic(path,obj)
    return obj
print('Inner CV ve referans model fonksiyonları hazır.')

## 5. Nested CV dış döngüsü: **108 iç eğitim + 6 dış-kat refit**

Bu hücre uzun sürebilir. Bütün sonuçlar öncelikle **development** verisinden üretilir. Her dış katta iki farklı arama prosedürü kendi en iyi adayını seçer, ardından sadece outer-train ile yeniden eğitilir ve outer evaluation'da bir kez ölçülür. Dummy ve Logistic Regression aynı dış değerlendirme katına uygulanır.

In [ ]:
outer_cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEEDS['outer_cv'])
outer_rows=[];all_inner=[]
for outer_k,(train_idx,eval_idx) in enumerate(outer_cv.split(X,y),1):
    phase=f'outer{outer_k}'
    print('\n###',phase,'başladı',flush=True)
    candidates=run_inner(phase,train_idx,SEEDS['inner_cv']+outer_k)
    all_inner.append(candidates)
    for method in ('manual','random'):
        ranking=candidates[candidates['method']==method].sort_values(
            ['mean_ap','candidate'],ascending=[False,True])
        best_id=ranking.iloc[0]['candidate'];cfg=ALL_CANDIDATES[best_id]
        print('Dış kat',outer_k,'yöntem',method,'seçilen',best_id,flush=True)
        rec=cached_train(f'{phase}_{method}_{best_id}_outer_refit',cfg,
            train_idx,eval_idx,seed_for(phase,0,variant=500),method)
        outer_rows.append(dict(outer_fold=outer_k,strategy=method,
            selected=best_id,inner_selected_mean_ap=float(ranking.iloc[0]['mean_ap']),**rec))
    for method in ('dummy','logreg'):
        rec=baseline_outer(method,train_idx,eval_idx,phase)
        outer_rows.append(dict(outer_fold=outer_k,strategy=method,selected=method,
                               inner_selected_mean_ap=np.nan,**rec))
    save_csv_atomic(pd.concat(all_inner,ignore_index=True),RESULTS/'outer_inner_search_results.csv')
    save_csv_atomic(pd.DataFrame(outer_rows),RESULTS/'outer_cv_results.csv')
    print('Dış kat',outer_k,'tamamlandı; sonuçlar Drive\'a kaydedildi.',flush=True)

outer_frame=pd.DataFrame(outer_rows)
assert len(outer_frame)==12  # 3 fold * (2 search procedure + 2 baseline)
measures=['ap','roc_auc','accuracy','precision','sensitivity','specificity','f1',
          'balanced_accuracy','train_seconds']
summary=[]
for method,g in outer_frame.groupby('strategy',sort=False):
    assert len(g)==3
    result={'strategy':method,'n_outer_folds':3}
    for col in measures:
        result[col+'_mean']=float(g[col].mean())
        result[col+'_sd']=float(g[col].std(ddof=1))
    summary.append(result)
summary_frame=pd.DataFrame(summary)
save_csv_atomic(summary_frame,RESULTS/'outer_cv_mean_sd.csv')
display(summary_frame.round(4))
print('AŞAMA 2 dış nested CV başarıyla tamamlandı.')

## 6. Nihai seçim: **tüm development üzerinde 36 yeni inner eğitim**

Son aday **outer skorlarına bakarak değil**, tüm development üzerinde yeni 3-fold inner CV'deki en yüksek mean AP sonucuna göre seçilir. Bu aşamada da bağımsız test kullanılmaz.

In [ ]:
print('Tam development iç CV başlıyor (36 eğitim)...',flush=True)
all_dev=np.arange(len(y),dtype=int)
final_search=run_inner('final',all_dev,SEEDS['inner_cv']+100)
assert len(final_search)==12
save_csv_atomic(final_search[final_search['method']=='manual'],RESULTS/'manual_search_results.csv')
save_csv_atomic(final_search[final_search['method']=='random'],RESULTS/'random_search_results.csv')
ordered=final_search.sort_values(['mean_ap','candidate'],ascending=[False,True])
chosen=ordered.iloc[0];chosen_cfg=ALL_CANDIDATES[chosen['candidate']]
final_cfg={'project':'YSA_31040_20261118','seed_base':SEEDS['base'],
    'method':str(chosen['method']),'candidate':str(chosen['candidate']),
    'config':chosen_cfg,'development_inner_mean_ap':float(chosen['mean_ap']),
    'development_inner_sd_ap':float(chosen['sd_ap']),
    'selection_policy':'highest full-development inner CV mean AP, candidate ID tie break',
    'selection_metric':'average_precision','threshold':0.5,
    'split_sha256':SPLIT_SHA,'raw_data_sha256':RAW_SHA,'stage2_plan_sha256':PLAN_ID,
    'primary_initialization_seed':int(SEEDS['torch']),
    'initialization_seeds':[int(SEEDS['torch']+k) for k in (0,101,202)],
    'test_status':'UNSEEN — DO NOT evaluate until locked predictions are committed',
    'code_version':CODE_VERSION}
freeze_json(CONFIGS/'final_hyperparameters.json',final_cfg)
display(final_search.sort_values('mean_ap',ascending=False).round(4))
print('NİHAİ GELİŞTİRME SEÇİMİ:',final_cfg['method'],final_cfg['candidate'],
      'mean AP',round(final_cfg['development_inner_mean_ap'],4))
print('Bağımsız test verisi kullanılmadı.')

## 7. Deney görselleri, son kontroller, SHA-256 manifesti

Outer Mean ± SD değerleri, iç kat AP seçimi ve nihai seçim birbirinden ayrı yorumlanmalıdır. Üç dış katın SD'si %95 güven aralığı değildir. Aşama 3 için gereken nihai seçim **henüz test edilmemiştir**.

In [ ]:
# Karşılaştırma grafiği — aynı 3-fold inner CV üzerinden 12 aday.
scores=pd.read_csv(RESULTS/'final_inner_search.csv')
fig,ax=plt.subplots(figsize=(12,5))
colors=['#2176AE' if q=='manual' else '#C17C20' for q in scores['method']]
ax.bar(scores['candidate'],scores['mean_ap'],yerr=scores['sd_ap'],
       capsize=3,color=colors,alpha=0.85)
ax.set(title='Nihai development inner CV — manuel vs rastgele AP',
       xlabel='Aday',ylabel='Ortalama Average Precision ± SD')
fig.tight_layout();fig.savefig(FIGURES/'05_manual_vs_random_ap.png',dpi=180)
plt.show()

# Bir örnek eğitimin eğitim/early-stop BCE ve LR eğrisi.
example=HISTORIES/'outer1_manual_M1_inner1.csv'
if example.exists():
    hist=pd.read_csv(example)
    fig,ax=plt.subplots(figsize=(10,4.5))
    ax.plot(hist['epoch'],hist['training_bce'],label='Fit — training BCE')
    ax.plot(hist['epoch'],hist['monitor_bce'],label='Early-stop monitor BCE')
    ax.set(xlabel='Epoch',ylabel='BCE kaybı',title='M1 örnek iç kat eğitim ve monitor kaybı')
    ax.legend();fig.tight_layout();fig.savefig(FIGURES/'06_training_monitor_loss.png',dpi=180)
    plt.show()
    fig,ax=plt.subplots(figsize=(10,3))
    ax.plot(hist['epoch'],hist['learning_rate'])
    ax.set(xlabel='Epoch',ylabel='Learning rate',title='M1 learning rate scheduler geçmişi')
    fig.tight_layout();fig.savefig(FIGURES/'07_learning_rate.png',dpi=180)
    plt.show()

# Aşama 3 kilitleme aşamasında dosya SHA'ları birebir kontrol edilecek.
final_check=[RESULTS/'outer_inner_search_results.csv',RESULTS/'outer_cv_results.csv',
    RESULTS/'outer_cv_mean_sd.csv',RESULTS/'final_inner_search.csv',
    RESULTS/'manual_search_results.csv',RESULTS/'random_search_results.csv',
    CONFIGS/'manual_candidates.json',CONFIGS/'random_candidates.json',
    CONFIGS/'training_policy.json',CONFIGS/'stage2_plan.json',
    CONFIGS/'final_hyperparameters.json']
assert all(p.is_file() for p in final_check), 'Aşama 2 eksik dosyaları var'
completed=[p for p in RUNS.glob('*.json') if not p.name.endswith('.failure.json')]
assert len([p for p in completed if '_inner' in p.name])==144, '144 inner eğitim sonucu bekleniyordu'
assert len([p for p in completed if '_outer_refit' in p.name])==6, '6 outer refit bekleniyordu'
assert len([p for p in completed if '_baseline_' in p.name])==6, '6 referans model bekleniyordu'
sha_manifest={'code_version':CODE_VERSION,'base_seed':31040,
 'stage2_plan_sha256':PLAN_ID,'split_sha256':SPLIT_SHA,
 'hash_algorithm':'SHA-256 of raw file bytes',
 'files':{str(p.relative_to(ROOT)):sha256_file(p)
    for p in sorted(list(CONFIGS.glob('*.json'))+list(RESULTS.glob('*.csv')))
    if p.is_file() and p.name!='stage2_sha256_manifest.json'}}
save_json_atomic(RESULTS/'stage2_sha256_manifest.json',sha_manifest)
print('AŞAMA 2 BAŞARILI — 108 outer-inner + 36 final-inner + 6 outer refit + 6 baseline doğrulandı.')
print('Nihai seçim:',CONFIGS/'final_hyperparameters.json')
print('Çıktılar:',ROOT)
print('Aşama 3: nihai modelleri eğit ve SHA + test tahmini kilitleme commit işlemini tamamla.')